In [7]:
import pandas as pd
import numpy as np

In [8]:
def get_engineers_info(control_df: pd.DataFrame) -> pd.DataFrame:
    _start = pd.to_datetime(control_df["Начало"], format="%d.%m.%Y %H:%M")
    _end = pd.to_datetime(control_df["Окончание"], format="%d.%m.%Y %H:%M")

    agg_kwargs = {
        "shift_start": ("_start", "min"),
        "shift_end": ("_end", "max"),
        "gigabit_connection": ("Гигабитное подключение", lambda s: "Да" in set(s.dropna().unique())),
    }

    if "Подключение" in control_df.columns:
        agg_kwargs["equipment_types"] = (
            "Подключение",
            lambda s: set(s.dropna().unique())
        )

    engineers_df = (
        control_df.assign(_start=_start, _end=_end)
        .dropna(subset=["name"])
        .groupby("name")
        .agg(**agg_kwargs)
        .reset_index()
        .sort_values("shift_start")
    )

    engineers_df["shift_start"] = engineers_df["shift_start"].dt.tz_localize("Europe/Moscow").dt.strftime(
        "%Y-%m-%dT%H:%M:%S%z")
    engineers_df["shift_end"] = engineers_df["shift_end"].dt.tz_localize("Europe/Moscow").dt.strftime(
        "%Y-%m-%dT%H:%M:%S%z")

    if "equipment_types" not in engineers_df.columns:
        engineers_df["equipment_types"] = [set() for _ in range(len(engineers_df))]

    def _build_equipment(row):
        equipment = set(row["equipment_types"]) if row["equipment_types"] else set()
        if row["gigabit_connection"]:
            equipment.add("gigabit_connection")
        return equipment or None

    engineers_df["equipment"] = engineers_df.apply(_build_equipment, axis=1)
    engineers_df = engineers_df.drop(columns=["equipment_types", "gigabit_connection"])

    return engineers_df


In [9]:
east_synthetic_df = pd.read_csv("./data/Восток Синтетические данные.csv", sep=';', encoding="windows-1251")
east_control_df = pd.read_csv("./data/Восток Контрольное распределение..csv", sep=';', encoding="windows-1251")
if "Бригада" in east_control_df.columns:
    east_control_df = east_control_df.rename(columns={"Бригада": "name"})

In [10]:
GROUP_KEYS = ["Начало", "Окончание", "Район"]

REQ_TO_SKILL_COLUMN = {
    "required_skill_local_works": "skill_local_works",
    "required_skill_connection_works": "skill_connection_works",
    "required_skill_emergency_works": "skill_emergency_works",
}


def assign_required_skills(requests_df, p_true=0.5, seed=None):
    rng = np.random.default_rng(seed)
    requests_df = requests_df.copy()
    n = len(requests_df)

    flags = rng.random((n, len(REQ_TO_SKILL_COLUMN))) < p_true
    empty = ~flags.any(axis=1)
    flags[empty, rng.integers(0, flags.shape[1], empty.sum())] = True  # at least one skill

    for i, col in enumerate(REQ_TO_SKILL_COLUMN):
        requests_df[col] = flags[:, i]
    return requests_df


def attach_skills_and_vehicle_to_control(control_df: pd.DataFrame, requests_df: pd.DataFrame) -> pd.DataFrame:
    if len(control_df) != len(requests_df):
        raise ValueError(f"{len(control_df)} rows in control_df vs {len(requests_df)} in requests_df.")

    control_df = control_df.reset_index(drop=True).copy()
    requests_reset = requests_df.reset_index(drop=True)

    # NaN != NaN in pandas, so fill with a sentinel before comparing —
    # otherwise "both missing" gets flagged as a mismatch
    _SENTINEL = "__MISSING__"
    control_keys = control_df[GROUP_KEYS].fillna(_SENTINEL)
    requests_keys = requests_reset[GROUP_KEYS].fillna(_SENTINEL)

    diff_mask = control_keys != requests_keys
    mismatches = diff_mask.any(axis=1)
    if mismatches.any():
        bad_idx = mismatches[mismatches].index.tolist()
        print(f"Row mismatch at {len(bad_idx)} position(s), showing up to 5:")
        for i in bad_idx[:5]:
            bad_cols = diff_mask.columns[diff_mask.loc[i]].tolist()
            for col in bad_cols:
                print(
                    f"  row {i} | column {col!r}: control={control_df.loc[i, col]!r} vs requests={requests_reset.loc[i, col]!r}")
        raise ValueError(f"Row mismatch at position(s) {bad_idx[:5]} — control_df and requests_df aren't aligned.")

    for req_col, skill_col in REQ_TO_SKILL_COLUMN.items():
        control_df[skill_col] = requests_reset[req_col]
    control_df["vehicle"] = requests_reset["required_vehicle"]
    return control_df


def fill_engineer_skills(engineers_df: pd.DataFrame, control_with_skills: pd.DataFrame) -> pd.DataFrame:
    """A brigade 'has' a skill if any ticket they were assigned required it."""
    engineers_df = engineers_df.copy()
    skill_cols = list(REQ_TO_SKILL_COLUMN.values())

    skill_by_brigade = (
        control_with_skills.dropna(subset=["name"])
        .groupby("name")[skill_cols]
        .any()
    )

    engineers_df = engineers_df.merge(skill_by_brigade, left_on="name", right_index=True, how="left")
    for col in skill_cols:
        engineers_df[col] = engineers_df[col].fillna(False)

    def _build_skills(row):
        skills = set()
        for col in skill_cols:
            if row[col]:
                skills.add(col)
        return skills

    engineers_df["skills"] = engineers_df.apply(_build_skills, axis=1)
    engineers_df = engineers_df.drop(columns=skill_cols)

    return engineers_df


VEHICLES = ["car", "walk", "bicycle", "public_transport"]


def assign_required_vehicles(
    requests_df: pd.DataFrame,
    control_df: pd.DataFrame,
    seed: int | None = None,
) -> pd.DataFrame:
    """One random vehicle per brigade; every ticket inherits its brigade's vehicle.
    Tickets without a brigade get a random vehicle."""
    if len(control_df) != len(requests_df):
        raise ValueError(f"{len(control_df)} rows in control_df vs {len(requests_df)} in requests_df.")

    rng = np.random.default_rng(seed)
    requests_df = requests_df.reset_index(drop=True).copy()
    names = control_df.reset_index(drop=True)["name"]

    brigades = names.dropna().unique()
    vehicle_by_brigade = dict(zip(brigades, rng.choice(VEHICLES, size=len(brigades))))

    vehicles = names.map(vehicle_by_brigade)
    unassigned = vehicles.isna()
    vehicles[unassigned] = rng.choice(VEHICLES, size=unassigned.sum())

    requests_df["required_vehicle"] = vehicles.to_numpy()
    return requests_df


WORK_TYPE_DURATION = {
    "connect_client": 90,
    "emergency_work": 100,
    "postorder": 40,
    "local_work_or_repair": 50,
}

WORK_TYPE_TO_SKILL = {
    "connect_client": "required_skill_connection_works",
    "emergency_work": "required_skill_emergency_works",
    "local_work_or_repair": "required_skill_local_works",
    "postorder": "required_skill_local_works",
}


def assign_work_type_and_skill(requests_df, weights: dict[str, float] | None = None, seed=None):
    """Random work_type per ticket; the ticket's single required skill follows from it."""
    rng = np.random.default_rng(seed)
    requests_df = requests_df.copy()

    types = list(WORK_TYPE_TO_SKILL)
    p = None
    print(weights)
    if weights is not None:
        w = np.array([weights[t] for t in types], dtype=float)
        p = w / w.sum()
    print(p)
    requests_df["work_type"] = rng.choice(types, size=len(requests_df), p=p)
    skill_col = requests_df["work_type"].map(WORK_TYPE_TO_SKILL)
    for col in REQ_TO_SKILL_COLUMN:  # keeps the boolean columns downstream code expects
        requests_df[col] = skill_col == col
    return requests_df


def assign_required_vehicles(
    requests_df: pd.DataFrame,
    control_df: pd.DataFrame,
    seed: int | None = None,
) -> pd.DataFrame:
    """One random vehicle per brigade; tickets inherit it. Walk is the default,
    so it's stored as None (no requirement)."""
    if len(control_df) != len(requests_df):
        raise ValueError(f"{len(control_df)} rows in control_df vs {len(requests_df)} in requests_df.")

    rng = np.random.default_rng(seed)
    requests_df = requests_df.reset_index(drop=True).copy()
    names = control_df.reset_index(drop=True)["name"]

    brigades = names.dropna().unique()
    vehicle_by_brigade = dict(zip(brigades, rng.choice(VEHICLES, size=len(brigades))))

    vehicles = names.map(vehicle_by_brigade)
    unassigned = vehicles.isna()
    vehicles[unassigned] = rng.choice(VEHICLES, size=unassigned.sum())

    requests_df["required_vehicle"] = vehicles.where(vehicles != "walk").to_numpy()
    return requests_df


def fill_engineer_vehicles(
    engineers_df: pd.DataFrame,
    control_with_skills: pd.DataFrame,
    seed: int | None = None,
) -> pd.DataFrame:
    """Engineer's vehicle = the vehicle required by their tickets; no requirement -> walk."""
    rng = np.random.default_rng(seed)
    engineers_df = engineers_df.copy()

    grouped = control_with_skills.dropna(subset=["name"]).groupby("name")["vehicle"]

    n_unique = grouped.nunique()  # ignores None, so walk tickets don't count as a conflict
    conflicts = n_unique[n_unique > 1]
    if not conflicts.empty:
        raise ValueError(f"Brigades with mixed vehicle requirements: {conflicts.index.tolist()[:5]}")

    has_tickets = engineers_df["name"].isin(grouped.groups.keys())
    engineers_df["vehicle"] = engineers_df["name"].map(grouped.first())

    # Has tickets but none require a vehicle -> walks
    engineers_df.loc[has_tickets & engineers_df["vehicle"].isna(), "vehicle"] = "walk"

    # No tickets at all -> random
    no_tickets = ~has_tickets
    engineers_df.loc[no_tickets, "vehicle"] = rng.choice(VEHICLES, size=no_tickets.sum())

    return engineers_df


DATE_FMT = "%d.%m.%Y %H:%M"
TZ = "Europe/Moscow"


def assign_duration(
    requests_df: pd.DataFrame,
) -> pd.DataFrame:
    """Duration from work_type (clipped to the time window); random priority."""
    requests_df = requests_df.copy()

    duration = requests_df["work_type"].map(WORK_TYPE_DURATION)
    requests_df["duration"] = duration
    return requests_df


def to_iso_msk(col: pd.Series) -> pd.Series:
    return (
        pd.to_datetime(col, format=DATE_FMT)
        .dt.tz_localize(TZ)
        .dt.strftime("%Y-%m-%dT%H:%M:%S%z")
    )


def build_required_equipment(row) -> set[str] | None:
    equipment = set()

    if "Подключение" in row and pd.notna(row["Подключение"]):
        equipment.add(row["Подключение"])
    if row.get("Гигабитное подключение") == "Да":
        equipment.add("gigabit_connection")
    return equipment or None


def build_request_status(row) -> str:
    if row["Статус BK"] == "Отправлена":
        return "sent"
    elif row["Статус BK"] == "Отменена":
        return "cancelled"
    elif row["Статус BK"] == "Выполнена":
        return "done"
    return "on_the_way"


def get_requests_info(requests_df: pd.DataFrame) -> pd.DataFrame:
    skill_cols = list(REQ_TO_SKILL_COLUMN)
    skills = requests_df[skill_cols].apply(
        lambda r: {REQ_TO_SKILL_COLUMN[c] for c in skill_cols if r[c]}, axis=1
    )
    vehicle = requests_df["required_vehicle"].astype(object)
    vehicle = vehicle.where(vehicle.notna(), None)

    return pd.DataFrame({
        "request_id": requests_df["Заявка"],
        "address": requests_df["Адрес"],
        "work_type": requests_df["work_type"],
        "duration": requests_df["duration"],
        "window_start": to_iso_msk(requests_df["Начало"]),
        "window_end": to_iso_msk(requests_df["Окончание"]),
        "required_skills": skills,
        "required_vehicle": vehicle,
        "status": requests_df.apply(build_request_status, axis=1),
        "required_equipment": requests_df.apply(build_required_equipment, axis=1),
    }).reset_index(drop=True)

In [11]:
def build_scenario(
    requests_df: pd.DataFrame,
    control_df: pd.DataFrame,
    office: str,
    seed: int | None = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Synthetic requests + control table -> (requests_info, engineers_info)."""
    requests_df = assign_work_type_and_skill(requests_df, seed=seed)
    requests_df = assign_duration(requests_df)
    requests_df = assign_required_vehicles(requests_df, control_df, seed=seed)
    control = attach_skills_and_vehicle_to_control(control_df, requests_df)
    print(control.columns)
    print(requests_df.columns)


    requests_df["Статус BK"] = control["Статус BK"].to_numpy()

    engineers = get_engineers_info(control)
    engineers = fill_engineer_skills(engineers, control)
    engineers = fill_engineer_vehicles(engineers, control, seed=seed)
    engineers["office"] = office

    return get_requests_info(requests_df), engineers

In [12]:
OFFICE = "г. Москва, ул Юных Ленинцев, д 83с 4"

east_requests_df, engineers_east_df = build_scenario(
    east_synthetic_df, east_control_df, office=OFFICE, seed=42
)

None
None
Index(['Заявка', 'Тип заявки BK', 'Статус BK', 'Тип заявки HD', 'Начало',
       'Окончание', 'Район', 'Адрес', 'name', 'Подключение',
       'Гигабитное подключение', 'skill_local_works', 'skill_connection_works',
       'skill_emergency_works', 'vehicle'],
      dtype='str')
Index(['Заявка', 'Тип заявки BK', 'Тип заявки HD', 'Начало', 'Окончание',
       'Район', 'Адрес', 'Подключение', 'Гигабитное подключение', 'work_type',
       'required_skill_local_works', 'required_skill_connection_works',
       'required_skill_emergency_works', 'duration', 'required_vehicle'],
      dtype='str')


In [13]:
engineers_east_df.to_csv("./data/EastEngineersData.csv", index=False)
east_requests_df.to_csv("./data/EastRequestsData.csv", index=False)
east_synthetic_df.to_csv("./data/EastSyntethicData.csv", index=False)
east_control_df.to_csv("./data/EastControlData.csv", index=False)

In [14]:
south_east_synthetic_df = pd.read_csv("./data/Юго-восток Синтетические данные.csv", sep=';', encoding="windows-1251")
south_east_control_df = pd.read_csv("./data/Юго-восток Контрольное распределение.csv", sep=';', encoding="windows-1251")
if "Бригада" in south_east_control_df.columns:
    south_east_control_df = south_east_control_df.rename(columns={"Бригада": "name"})

In [15]:
OFFICE = "г. Москва, ул Бирюлёвская, д 1с1"

south_east_requests_df, south_east_engineers_df = build_scenario(
    south_east_synthetic_df, south_east_control_df, office=OFFICE, seed=42
)

None
None
Index(['Заявка', 'Тип заявки BK', 'Статус BK', 'Тип заявки HD', 'Начало',
       'Окончание', 'Район', 'Адрес', 'name', 'Гигабитное подключение',
       'skill_local_works', 'skill_connection_works', 'skill_emergency_works',
       'vehicle'],
      dtype='str')
Index(['Заявка', 'Тип заявки BK', 'Тип заявки HD', 'Начало', 'Окончание',
       'Район', 'Адрес', 'Гигабитное подключение', 'work_type',
       'required_skill_local_works', 'required_skill_connection_works',
       'required_skill_emergency_works', 'duration', 'required_vehicle'],
      dtype='str')


In [16]:
south_east_engineers_df.to_csv("./data/SouthEastEngineersData.csv", index=False)
south_east_requests_df.to_csv("./data/SouthEastRequestsData.csv", index=False)
south_east_synthetic_df.to_csv("./data/SouthEastSyntethicData.csv", index=False)
east_control_df.to_csv("./data/SouthEastControlData.csv", index=False)

In [17]:
south_center_synthetic_df = pd.read_csv("./data/ЮгоЦентр Синтетические данные.csv", sep=';', encoding="windows-1251")
south_center_control_df = pd.read_csv("./data/Югоцентр Контрольное распределение..csv", sep=';',
                                      encoding="windows-1251")
if "Бригада" in south_center_control_df.columns:
    south_center_control_df = south_center_control_df.rename(columns={"Бригада": "name"})

In [18]:
OFFICE = "г.Москва проезд Симферопольский, д.7"

south_center_requests_df, south_center_east_engineers_df = build_scenario(
    south_center_synthetic_df, south_center_control_df, office=OFFICE, seed=42
)

None
None
Index(['Заявка', 'Тип заявки BK', 'Статус BK', 'Тип заявки HD', 'Начало',
       'Окончание', 'Район', 'Адрес', 'name', 'Гигабитное подключение',
       'skill_local_works', 'skill_connection_works', 'skill_emergency_works',
       'vehicle'],
      dtype='str')
Index(['Заявка', 'Тип заявки BK', 'Тип заявки HD', 'Начало', 'Окончание',
       'Район', 'Адрес', 'Гигабитное подключение', 'work_type',
       'required_skill_local_works', 'required_skill_connection_works',
       'required_skill_emergency_works', 'duration', 'required_vehicle'],
      dtype='str')


In [19]:
south_center_synthetic_df.to_csv("./data/SouthCenterSyntethicData.csv", index=False)
south_center_requests_df.to_csv("./data/SouthCenterRequestsData.csv", index=False)
south_center_control_df.to_csv("./data/SouthCenterControlData.csv", index=False)
south_center_east_engineers_df.to_csv("./data/SouthCenterEngineersData.csv", index=False)